**Summary At The End Of The NoteBook**

In [ ]:
import sys

# If in Colab, install the necessary tools
if 'google.colab' in sys.modules:
    !pip install gdown duckdb openpyxl pandas numpy
    


* **Importing Python Libraries.**
* **Appending Two Excel Sheets Into Single CSV File.**
* **Export Combined Excel Sheets Into One CSV File.**
* **Initialize DB Connection.**


In [ ]:
import pandas as pd
import duckdb
import numpy as np  # Added for handling numerical operations



import gdown

# Google Drive file ID
file_id = "1-V_GXHZhqLW4xpBR4uLMtR820txoijxh"

# Destination path (local filename)
FILE_PATH = "online_retail_II.xlsx"

# Download the file

url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=xlsx"
gdown.download(url, FILE_PATH, quiet=False)



# 1. Define all file paths in one place

CSV_PATH = "combined.csv"
DB_PATH = "combined.duckdb"

# 2. Load and Combine Sheets
sheet1 = pd.read_excel(FILE_PATH, sheet_name=0)
sheet2 = pd.read_excel(FILE_PATH, sheet_name=1)
combined = pd.concat([sheet1, sheet2], ignore_index=True)

# 3. Export to CSV using the relative path (more portable than absolute paths)
combined.to_csv(CSV_PATH, index=False, encoding="utf-8")

# 4. Initialize Database Connection
con = duckdb.connect(DB_PATH)
print("Setup complete. Data combined and Database connected.")

Create Pandas DataFrame

In [ ]:

df = pd.read_csv(
    CSV_PATH,
    encoding="utf-8"
)


Checking DATA

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:

total_rows = len(df)
unique_invoices = df["Invoice"].nunique()

print("Total rows:", total_rows)
print("Unique InvoiceNo:", unique_invoices)

In [ ]:
df.duplicated().sum()

Checking IF IT Work

In [ ]:
con.execute(f"""
    SELECT COUNT(*) AS total_rows
    FROM read_csv_auto(
        '{CSV_PATH}')
""").fetchone()

Creating A Sales Table From The DataSet

In [ ]:
con.execute(f"""
CREATE TABLE IF NOT EXISTS Sales AS
SELECT
    row_number() OVER () AS retail_row_id,
    *
FROM read_csv_auto('{CSV_PATH}')
""")

Checked Table Sales

In [ ]:
con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM Sales
    
""").fetchone()

In [ ]:
con.execute("""SELECT * FROM Sales LIMIT 5""").fetchdf()

Checking DataTypes And Constrains

In [ ]:
columns = con.execute("PRAGMA table_info('Sales');").fetchdf()
print(columns)

Make Some Changes

In [ ]:
con.execute("""ALTER TABLE Sales RENAME COLUMN "Customer ID" TO CustomerID;
ALTER TABLE Sales RENAME COLUMN Invoice TO InvoiceNo;
ALTER TABLE Sales RENAME COLUMN Price TO UnitPrice;
ALTER TABLE Sales RENAME COLUMN "retail_row_id" TO RetailRowID;

ALTER TABLE Sales ALTER COLUMN CustomerID SET DATA TYPE VARCHAR;
""")

Checking Nullability

In [ ]:
con.execute("""SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN RetailRowID IS NULL THEN 1 ELSE 0 END) AS null_retail_row_id,
    SUM(CASE WHEN InvoiceNo IS NULL THEN 1 ELSE 0 END) AS null_InvoiceNo,
    SUM(CASE WHEN StockCode IS NULL THEN 1 ELSE 0 END) AS null_StockCode,
    SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_Description,
    SUM(CASE WHEN Quantity IS NULL THEN 1 ELSE 0 END) AS null_Quantity,
    SUM(CASE WHEN InvoiceDate IS NULL THEN 1 ELSE 0 END) AS null_InvoiceDate,
    SUM(CASE WHEN UnitPrice IS NULL THEN 1 ELSE 0 END) AS null_UnitPrice,
    SUM(CASE WHEN "CustomerID" IS NULL THEN 1 ELSE 0 END) AS null_CustomerID,
    SUM(CASE WHEN Country IS NULL THEN 1 ELSE 0 END) AS null_Country
FROM Sales;""").fetchdf ()
  

Check For Duplicate Line Items

In [ ]:
con.execute('''
SELECT InvoiceNo, StockCode, COUNT(*) AS cnt
FROM Sales
GROUP BY InvoiceNo, StockCode
HAVING cnt > 1
 ''').fetchdf()

Remove Duplicated Rows

In [ ]:
con.execute('''
CREATE OR REPLACE TABLE Sales AS 
SELECT * FROM Sales 
QUALIFY row_number() OVER (
    PARTITION BY InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country
) = 1;
''')

Check For Removing Duplicates

In [ ]:
con.execute('''
SELECT *
FROM Sales
QUALIFY row_number() OVER (
    PARTITION BY 
        InvoiceNo, 
        StockCode, 
        Description, 
        Quantity, 
        InvoiceDate, 
        UnitPrice, 
        CustomerID, 
        Country
) > 1;
''').fetchdf()

Count Rows With Quantity Of Zero OR Less

In [ ]:
con.execute('''SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN Quantity <= 0 THEN 1 ELSE 0 END) AS non_positive_quantity
FROM Sales;''').fetchdf()


###
UnitPrice < 0 
Financial Adjustments


In [ ]:
con.execute('''select * from SALES
Where Unitprice < 0
limit 20''').fetchdf()

Flagged Them As Adjustments

In [ ]:

con.execute("ALTER TABLE Sales ADD COLUMN IF NOT EXISTS IsAdjustment BOOLEAN DEFAULT FALSE;")


con.execute("""
    UPDATE Sales
    SET IsAdjustment = TRUE
    WHERE StockCode = 'B' 
    AND Description = 'Adjust bad debt';
""")

Negative Quantities Due to Cancelled Invoices


In [ ]:
con.execute('''select * from SALES
Where InvoiceNo LIKE 'C%'
''').fetchdf()

Flagged Them as Cancellations

In [ ]:
con.execute('''ALTER TABLE Sales ADD COLUMN IsCancelled BOOLEAN DEFAULT FALSE;

UPDATE Sales
SET IsCancelled = TRUE
Where InvoiceNo LIKE 'C%';''')

Negative Quantities Due to Special Cases


In [ ]:
con.execute('''select count(*) from SALES
Where Quantity < 0 and IsCancelled = FALSE and UnitPrice = 0
''').fetchdf()

Add A Boolean Flag IsSpecialCase

In [ ]:
con.execute('''ALTER TABLE Sales ADD COLUMN IsSpecialCase BOOLEAN DEFAULT FALSE;''')


Update Data

In [ ]:
con.execute('''UPDATE Sales    
SET IsSpecialCase = TRUE
WHERE Quantity < 0
  AND IsCancelled = FALSE
  AND UnitPrice <=0;''')

Checking Ranges of CustomerID

In [ ]:
con.execute('''select min(CustomerID), max(CustomerID) from Sales''').fetchdf()

Number Of CustomerID Nulls

In [ ]:
con.execute('''select count(*) from Sales where CustomerID IS NULL
            ''').fetchdf()

Auditing The source of CustomerID Nulls

In [ ]:
con.execute('''
            SELECT
    -- Cancellations
    SUM(CASE WHEN IsCancelled = TRUE THEN 1 ELSE 0 END) AS total_cancellations,
    SUM(CASE WHEN IsCancelled= TRUE AND CustomerID IS NULL THEN 1 ELSE 0 END) AS nulls_in_cancellations,

    -- Adjustments
    SUM(CASE WHEN IsAdjustment = TRUE THEN 1 ELSE 0 END) AS total_adjustments,
    SUM(CASE WHEN IsAdjustment = TRUE AND CustomerID IS NULL THEN 1 ELSE 0 END) AS nulls_in_adjustments,

    -- Special Cases
    
     SUM(CASE WHEN IsSpecialCase = TRUE THEN 1 ELSE 0 END) AS total_special_cases,
    SUM(CASE WHEN IsSpecialCase = TRUE AND CustomerID IS NULL THEN 1 ELSE 0 END) AS nulls_in_special_cases,

    -- Other rows (normal sales)
    SUM(CASE WHEN IsCancelled= FALSE AND IsAdjustment = FALSE AND IsSpecialCase = FALSE AND CustomerID IS NULL THEN 1 ELSE 0 END) AS nulls_in_other_rows
FROM Sales;

            
            ''').fetchdf()

Mark Anonymous Customers As -1

In [ ]:
con.execute('''UPDATE Sales
SET CustomerID = -1
WHERE CustomerID IS NULL
and IsAdjustment = FALSE
and IsSpecialCase = FALSE;''')

Cheking Country Names

In [ ]:
con.execute('''select distinct Country from Sales;''').fetchdf()


Standarize Country Names

In [ ]:
con.execute('''
UPDATE Sales
SET Country = CASE
    WHEN Country = 'EIRE' THEN 'Ireland'
    WHEN Country = 'RSA' THEN 'South Africa'
    ELSE Country
END;
''')


Create A Country Dimension

In [ ]:
con.execute('''
CREATE TABLE dim_country AS
SELECT
    ROW_NUMBER() OVER (ORDER BY Country) AS CountryID,
    Country,
    CASE
        WHEN Country IN ('Unspecified', 'European Community', 'West Indies', 'Channel Islands')
        THEN FALSE ELSE TRUE
    END AS IsCountry
FROM (
    SELECT DISTINCT Country
    FROM Sales
);
''')


Checking The New Country Table

In [ ]:
con.execute('''SELECT * FROM dim_country ;''').fetchdf()


Add A New Column To Sales Fact Named CountryID

In [ ]:
con.execute('ALTER TABLE Sales ADD COLUMN CountryID INTEGER;')


Inserting The Data In CountryID

In [ ]:
con.execute('''
UPDATE Sales s
SET CountryID = c.CountryID
FROM dim_country c
WHERE s.Country = c.Country;
''')


Checking The New Column

In [ ]:
con.execute('''SELECT * FROM Sales LIMIT 5''').fetchdf()


Validating InvoiceNo

In [ ]:
con.execute('''
SELECT
    COUNT(*) AS total_rows,

    -- Null / empty / whitespace-only
    SUM(
        CASE 
            WHEN InvoiceNo IS NULL OR TRIM(InvoiceNo) = '' 
            THEN 1 ELSE 0 
        END
    ) AS null_or_blank,

    -- Leading or trailing spaces
    SUM(
        CASE 
            WHEN InvoiceNo IS NOT NULL AND InvoiceNo != TRIM(InvoiceNo)
            THEN 1 ELSE 0
        END
    ) AS leading_or_trailing_spaces,

    -- Internal spaces (invalid)
    SUM(
        CASE 
            WHEN InvoiceNo LIKE '% %'
            THEN 1 ELSE 0
        END
    ) AS internal_spaces,

    -- Invalid format (not 6 digits and not cancellation)
    SUM(
        CASE
            WHEN InvoiceNo IS NOT NULL
             AND TRIM(InvoiceNo) != ''
             AND NOT (
                    InvoiceNo LIKE 'C%' 
                 OR regexp_matches(InvoiceNo, '^[0-9]{6}$')
             )
            THEN 1 ELSE 0
        END
    ) AS invalid_format
FROM Sales;
''').fetch_df()


Investigating Invalid Format InvoiceNo Starts With A and They Are Adjustments(Already Flagged)

In [ ]:
con.execute('''
SELECT
        RetailRowID,
        InvoiceNo,
        StockCode,
        Description,
        Quantity,
        InvoiceDate,
        UnitPrice,
        CustomerID,
        Country
FROM Sales
WHERE InvoiceNo NOT 
    LIKE 'C%'
    AND NOT regexp_matches(InvoiceNo, '^[0-9]{6}$')
''').fetchdf()


Checking Stanadarization

In [ ]:
con.execute('''
SELECT
    -- Check for leading/trailing spaces
    SUM(CASE WHEN InvoiceNo != TRIM(InvoiceNo) THEN 1 ELSE 0 END) AS has_spaces_to_trim,
    
    -- Check for internal spaces (e.g., '536 365')
    SUM(CASE WHEN InvoiceNo LIKE '% %' THEN 1 ELSE 0 END) AS internal_spaces,
    
    -- Check for lower-case letters (e.g., 'c536365')
    -- Standard is 'C' for credit or purely numeric
    SUM(CASE WHEN InvoiceNo ~ '[a-z]' THEN 1 ELSE 0 END) AS has_lowercase,
    
    -- Check for non-alphanumeric characters (dots, dashes, etc.)
    SUM(CASE WHEN InvoiceNo ~ '[^a-zA-Z0-9]' THEN 1 ELSE 0 END) AS has_symbols
FROM Sales;
''').fetchdf()

Standarize StockCode

In [ ]:
con.execute("UPDATE Sales SET StockCode = UPPER(TRIM(StockCode));")

Remove Internal Space

In [ ]:
con.execute("UPDATE Sales SET StockCode = replace(StockCode, ' ', '');")

Check For Different Values

In [ ]:
con.execute('''SELECT
    SUM(CASE WHEN StockCode IS NULL OR TRIM(StockCode) = '' THEN 1 ELSE 0 END) AS empty,
    SUM(CASE WHEN StockCode != TRIM(StockCode) THEN 1 ELSE 0 END) AS leading_trailing_spaces,
    SUM(CASE WHEN LENGTH(StockCode) > 5 THEN 1 ELSE 0 END) AS more_than_5_chars,
    SUM(CASE WHEN StockCode ~ '([0-9].*){6,}' THEN 1 ELSE 0 END) AS more_than_5_digits,
    SUM(CASE WHEN StockCode ~ '.*[a-zA-Z].*[a-zA-Z].*' THEN 1 ELSE 0 END) AS more_than_2_letters,
    SUM(CASE WHEN StockCode LIKE '% %' THEN 1 ELSE 0 END) AS internal_spaces
FROM Sales;
''').fetchdf()


Creating StockType To Ensure Valid Products

In [ ]:
con.execute("ALTER TABLE Sales ADD COLUMN IF NOT EXISTS  StockType VARCHAR ;")


Classification Of Real Valid Products 

In [ ]:
con.execute('''
UPDATE Sales
SET StockType = CASE
    WHEN upper(StockCode) LIKE 'TEST%' THEN 'TEST'
    WHEN upper(StockCode) LIKE 'GIFT%' THEN 'GIFT_VOUCHER'
    WHEN upper(StockCode) IN ('ADJUST','ADJUST2','BANK CHARGES','AMAZONFEE','CRUK','B') THEN 'FINANCE'
    WHEN upper(StockCode) IN ('POST','DOT','PADS') THEN 'SERVICE'
    ELSE 'PRODUCT'
END;
''')

Count Null Descriptions That Have No Special Cases

In [ ]:

con.execute(""" 
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_descriptions,
    SUM(CASE WHEN IsSpecialCase = TRUE AND Description IS NULL THEN 1 ELSE 0 END) AS null_descriptions_special_case
FROM Sales

""").fetchdf()



We Have To Exclude This Invalid Rows From Further Analysis Because They Are Not A Purchase Behavior

In [ ]:
con.execute('''SELECT * FROM SALES WHERE Description IS NULL AND IsSpecialCase = FALSE AND CustomerID IS NOT NULL AND StockType = 'PRODUCT';''').fetchdf()


Checking MIN And MAX DATE

In [ ]:
con.execute('SELECT min(InvoiceDate), max(InvoiceDate) FROM Sales;').fetch_df()


Check IF One StockCode Point To More Than One ProductDescription

In [ ]:
con.execute('''
SELECT StockCode, 
    COUNT(DISTINCT UPPER(TRIM(Description))) AS description_count FROM Sales
WHERE NOT (
    Description IS NULL 
    AND IsSpecialCase = FALSE 
    AND CustomerID IS NOT NULL 
    AND StockType = 'PRODUCT' 
    AND IsAdjustment = FALSE
)
GROUP BY StockCode
HAVING COUNT(DISTINCT UPPER(TRIM(Description))) > 1
ORDER BY description_count DESC;
''').fetch_df   ()


Checking Which Description Will Be Assigned To One StockCode Given Number Of Frequencies And Last InvoiceDate

In [ ]:
con.execute('''WITH description_stats AS (
    SELECT
        StockCode,
        UPPER(TRIM(Description)) AS Description,
        COUNT(*) AS usage_count,
        MAX(InvoiceDate) AS last_seen
    FROM Sales
    WHERE
        Description IS NOT NULL
        AND StockType = 'PRODUCT'
        AND IsAdjustment = FALSE
        and IsSpecialCase = FALSE
        and CustomerID IS NOT NULL
    GROUP BY
        StockCode,
        UPPER(TRIM(Description))
),
ranked_descriptions AS (
    SELECT
        StockCode,
        Description,
        usage_count,
        last_seen,
        ROW_NUMBER() OVER (
            PARTITION BY StockCode
            ORDER BY
                last_seen DESC,        -- Rule 1: most recent
                usage_count DESC,      -- Rule 2: most frequent
                LENGTH(Description) DESC -- Rule 3: most descriptive
        ) AS chosen_rank,
        COUNT(*) OVER (PARTITION BY StockCode) AS description_count
    FROM description_stats
)
SELECT
    StockCode,
    Description,
    usage_count,
    last_seen,
    chosen_rank,
    description_count
FROM ranked_descriptions
WHERE description_count > 1
ORDER BY StockCode, chosen_rank;''').fetchdf()



Creating Product Dimension For Further Analysis

In [ ]:
# 1. Create a sequence
con.execute("CREATE SEQUENCE prod_key_seq START 1;")

# 2. Use the sequence in your table
con.execute('''
CREATE TABLE dim_Product (
    ProductKey INTEGER DEFAULT nextval('prod_key_seq'),
    StockCode VARCHAR,
    Description VARCHAR
);
''')

Filling The dim_product

In [ ]:
con.execute('''INSERT INTO dim_product (StockCode, Description)
WITH desc_stats AS (
    SELECT
        StockCode,
        UPPER(TRIM(Description)) AS Description,
        COUNT(*) AS usage_count,
        MAX(InvoiceDate) AS last_seen
    FROM Sales
    WHERE
         Description IS NOT NULL
        AND StockType = 'PRODUCT'
        AND IsAdjustment = FALSE
        and IsSpecialCase = FALSE
        and CustomerID IS NOT NULL
    GROUP BY StockCode, UPPER(TRIM(Description))
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY StockCode
            ORDER BY usage_count DESC, last_seen DESC, Description
        ) AS rn
    FROM desc_stats
)
SELECT
    StockCode,
    Description AS ProductDescription
    
FROM ranked
WHERE rn = 1; ''')

Validation Our Results

In [ ]:
con.execute('''select count(distinct StockCode) from Sales
            WHERE 
    Description IS  not NULL 
    AND IsSpecialCase = false
    AND CustomerID IS  not null
    AND StockType = 'PRODUCT' 
    AND IsAdjustment = FALSE
''').fetchdf()

In [ ]:
con.execute('''select count(StockCode) from dim_product ''').fetchdf()

In [ ]:
con.execute('''SELECT * FROM dim_product LIMIT 10;''').fetchdf()


Create Date Dimension 

In [ ]:
con.execute('''CREATE Or Replace TABLE dim_date AS
WITH date_bounds AS (
    SELECT 
        -- Exact min and max InvoiceDate from Sales
        CAST(MIN(InvoiceDate) AS DATE) AS start_date, 
        CAST(MAX(InvoiceDate) AS DATE) AS end_date
    FROM Sales
),
date_series AS (
    SELECT 
        generate_series AS full_date
    FROM date_bounds, 
         generate_series(start_date, end_date, interval '1 day')
)
SELECT
    CAST(strftime(full_date, '%Y%m%d') AS INTEGER) AS DateKey,
    full_date AS FullDate,
    year(full_date) AS Year,
    quarter(full_date) AS Quarter,
    month(full_date) AS Month,
    monthname(full_date) AS MonthName,
    -- MonthKey for chronological sorting (e.g., 201012)
    (year(full_date) * 100 + month(full_date)) AS MonthKey,
    week(full_date) AS WeekNumber,
    day(full_date) AS Day,
    dayname(full_date) AS DayName,
    dayofweek(full_date) AS DayOfWeek,
    CASE WHEN dayofweek(full_date) IN (0, 6) THEN TRUE ELSE FALSE END AS IsWeekend
FROM date_series;
''')


Add DateKey Column In Sales Table

In [ ]:
con.execute('ALTER TABLE Sales ADD COLUMN InvoiceDateKey INTEGER;')


In [ ]:
con.execute('ALTER TABLE Sales RENAME COLUMN InvoiceDateKey TO DateKey;')


Extract Date And Add It To DateKey As Integer

In [ ]:
con.execute('''UPDATE Sales
SET DateKey = CAST(strftime(InvoiceDate, '%Y%m%d') AS INTEGER); ''')

In [ ]:
con.execute('SELECT max(FullDate) FROM dim_date;').fetchdf()


Added ProductID To Sales

In [ ]:

con.execute("ALTER TABLE Sales ADD COLUMN IF NOT EXISTS  ProductID INTEGER;")


In [ ]:
con.execute("ALTER TABLE dim_product Rename COLUMN ProductKey to ProductID ;")

In [ ]:
con.execute('''
    UPDATE Sales
    SET ProductID = dim_product.ProductID
    FROM dim_product 
    WHERE Sales.StockCode = dim_product.StockCode;
''')

Final Cleaned Query

In [ ]:
con.execute('''SELECT sum(UnitPrice) FROM Sales
   where StockType = 'PRODUCT'
   and IsAdjustment = FALSE
   and IsSpecialCase = FALSE
   and CustomerID IS NOT NULL
   and Description IS not NULL  
   
    
         
         
    ;''').fetchdf()

Transform Cleaned Data Into New Table

In [ ]:
con.execute('''
CREATE TABLE fact_sales AS
SELECT 
    RetailRowID,
    InvoiceNo,
    StockCode,
    ProductID,     
    Quantity,
    InvoiceDate,
    UnitPrice,
    CustomerID,
    IsCancelled,
    CountryID,
    DateKey        
FROM Sales
WHERE StockType = 'PRODUCT'
  AND IsAdjustment = FALSE
  AND IsSpecialCase = FALSE
  AND CustomerID IS NOT NULL
  AND Description IS NOT NULL;
''')

Rename RetailRowID To InvoiceDetailID For More Clarity

In [ ]:
con.execute('ALTER TABLE fact_sales RENAME COLUMN RetailRowID TO InvoiceDetailID;')


UnitPrice = 0 Customer received a Damaged/Wrong Item

In [ ]:
con.execute("""
        SELECT 
           *
        FROM fact_sales  
        WHERE UnitPrice = 0 
        
    """).fetch_df()

Add A Flag For Revenue

In [ ]:
con.execute("""
ALTER TABLE fact_sales
ADD IsRevenue BOOLEAN;

UPDATE fact_sales
SET IsRevenue = 
    CASE 
        WHEN UnitPrice = 0 THEN FALSE
        ELSE TRUE
    END;
""")


Additional Cleaning for Cleaning For StockCode In Both Fact_sales and dim_product

In [ ]:
con.execute("""
DELETE FROM fact_sales
WHERE StockCode IN ('M', 'D', 'S', 'C2', 'BANKCHARGES',  '23595', '35600A')
OR StockCode LIKE 'DCGS%';
""")


In [ ]:
con.execute("""
DELETE FROM dim_product
WHERE StockCode IN ('M', 'D', 'S', 'C2', 'BANKCHARGES',  '23595', '35600A')
OR StockCode LIKE 'DCGS%';
""")


Checking Outliers : Become More Normal After Removing Internal Adjustments And Finances But Still Right Skewed

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns



# 1. Fetch the raw data (no need for fences in SQL)
df = con.execute("SELECT UnitPrice FROM fact_sales WHERE IsRevenue = true").fetch_df()

# 2. Create the boxplot
# 'whis=1.5' is the default, which matches your SQL logic exactly
sns.boxplot(x=df['UnitPrice'], color='skyblue', flierprops={'markerfacecolor':'red'})

plt.title('UnitPrice Outlier Detection')
plt.xlabel('Unit Price')
plt.grid(axis='x', linestyle='--', alpha=0.5)



Mean = 3.36 
Median = 2.1 
Due To Positive Skewness

In [ ]:
df =con.execute("""select UnitPrice from fact_sales where IsRevenue = true""").fetch_df()


In [ ]:
df.median()

In [ ]:
df.mean()

Checking Outliers For Quantity 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


# 1. Fetch the raw data (no need for fences in SQL)
df = con.execute("SELECT Quantity FROM fact_sales WHERE IsRevenue = true").fetch_df()

# 2. Create the boxplot
# 'whis=1.5' is the default, which matches your SQL logic exactly
sns.boxplot(x=df['Quantity'], color='skyblue', flierprops={'markerfacecolor':'red'})

plt.title('Quantity Outlier Detection')
plt.xlabel('Quantity')
plt.grid(axis='x', linestyle='--', alpha=0.5)


Measure Quantity Outlier Threshold For Sales

In [ ]:
con.execute('''WITH stats AS (
    SELECT
        quantile_cont(Quantity, 0.25) AS Q1,
        quantile_cont(Quantity, 0.75) AS Q3
    FROM fact_sales
    WHERE Quantity > 0
      AND IsCancelled = FALSE
)
SELECT
    Q3 + 1.5 * (Q3 - Q1) AS bulk_sale_threshold
FROM stats;''').fetch_df()

Measure Quantity Outlier Threshold For Cancellations

In [ ]:
con.execute('''WITH stats AS (
    SELECT
        quantile_cont(ABS(Quantity), 0.25) AS Q1,
        quantile_cont(ABS(Quantity), 0.75) AS Q3
    FROM fact_sales
    WHERE Quantity < 0
      AND IsCancelled = TRUE
)
SELECT
    Q3 + 1.5 * (Q3 - Q1) AS cancellation_bulk_threshold
FROM stats;
''').fetch_df()

Add Flags For Transactions Over Thresholds 

In [ ]:
con.execute('''ALTER TABLE fact_sales ADD COLUMN IsBulkSale BOOLEAN DEFAULT FALSE;
ALTER TABLE fact_sales ADD COLUMN IsBulkCancellation BOOLEAN DEFAULT FALSE;''')

In [ ]:
con.execute('''UPDATE fact_sales
SET IsBulkSale = TRUE
WHERE Quantity >= 29''')
  

In [ ]:
con.execute('''UPDATE fact_sales
SET IsBulkCancellation  = TRUE
where Quantity <=-14''')

Add Flag To Know Which Customers Are Known And Add Them To Customer Dimension And For Easier Slicing In PowerBI

In [ ]:
con.execute('''ALTER TABLE fact_sales ADD COLUMN IF NOT EXISTS IsAnonymous BOOLEAN DEFAULT FALSE;''')

Update Data

In [ ]:
con.execute('''UPDATE fact_sales
SET IsAnonymous = TRUE
WHERE CustomerID = -1
 ''')

Add Flag For Overlapping Between Revenue And Non-Revenue Due To Different Reasons Damaged Goods / Order Correction etc...

In [ ]:
con.execute('''ALTER TABLE fact_sales
ADD COLUMN IF NOT EXISTS  HasRevenueNonRevenueOverlap BOOLEAN DEFAULT FALSE;''')

OverAll NonRevenue rows are 959 And 899 Related To Anonymous Customers and 60 For Known Customers And 47 For Known Customers With Prior Order Revenue And Then  Order With No Revenue (Overlapping) And Those What We Need During Analysis

In [ ]:
con.execute('''UPDATE fact_sales AS r
SET HasRevenueNonRevenueOverlap = TRUE
FROM fact_sales AS nr
WHERE r.CustomerID = nr.CustomerID
  AND r.StockCode = nr.StockCode
  AND r.Quantity = nr.Quantity
  AND r.IsRevenue = TRUE
  AND nr.IsRevenue = FALSE
  AND r.IsAnonymous = FALSE
  AND nr.InvoiceDate > r.InvoiceDate
  AND r.UnitPrice != nr.UnitPrice;''')

Creating Customer Dimension With CustomerType And FirstPurchaseDate And LastPurchaseDate And IsRepeatCustomer

In [ ]:
con.execute('''CREATE OR REPLACE TABLE dim_customer AS
WITH customer_activity AS (
    SELECT
        CustomerID,
        -- FIX 1: Count UNIQUE Invoices (Actual visits to the store)
        COUNT(DISTINCT CASE 
                WHEN Quantity > 0 AND IsCancelled = FALSE 
                THEN InvoiceNo 
            END) AS unique_purchase_orders,

        -- FIX 2: Count UNIQUE Cancellations
        COUNT(DISTINCT CASE 
                WHEN IsCancelled = TRUE 
                THEN InvoiceNo 
            END) AS unique_cancel_orders,

        -- First & last successful purchase dates
        MIN(CASE 
                WHEN Quantity > 0 AND IsCancelled = FALSE 
                THEN InvoiceDate 
            END) AS FirstPurchaseDate,

        MAX(CASE 
                WHEN Quantity > 0 AND IsCancelled = FALSE 
                THEN InvoiceDate 
            END) AS LastPurchaseDate

    FROM fact_sales
    WHERE CustomerID != -1   
    -- REMOVED: IsRevenue = TRUE (to ensure we see all cancellation attempts)
    GROUP BY CustomerID
)

SELECT
    CustomerID,
    CASE
        WHEN unique_purchase_orders > 0 AND unique_cancel_orders = 0 THEN 'OnlyPurchases'
        WHEN unique_purchase_orders > 0 AND unique_cancel_orders > 0 THEN 'PurchasesAndCancellations'
        WHEN unique_purchase_orders = 0 AND unique_cancel_orders > 0 THEN 'OnlyCancellations'
        ELSE 'InquiryOnly'
    END AS CustomerType,
    FirstPurchaseDate,
    LastPurchaseDate,
    -- FIX 3: Repeat flag based on VISITS, not items
    CASE 
        WHEN unique_purchase_orders > 1 THEN TRUE
        ELSE FALSE
    END AS IsRepeatCustomer
FROM customer_activity;''')

Country Is Defined In The DataSource As The Residence Country For Customers

In [ ]:
con.execute('''SELECT 
    CustomerID, 
    COUNT(DISTINCT CountryID) AS country_count
FROM fact_sales
WHERE IsAnonymous = FALSE
GROUP BY CustomerID
HAVING country_count > 1;''').fetch_df()

We Added RecentCountryID Since There Are Customers Point To More Than One Country.

We Did This For Easier Customer-Level Country Analysis 

In [ ]:
con.execute('''ALTER TABLE dim_customer
ADD COLUMN IF NOT EXISTS RecentCountryID INT;''')

Populated The RecentCountryID With The Country In The Most Recent Transaction Each Customer Has Done.

In [ ]:
con.execute('''WITH recent_country AS (
    SELECT
        CustomerID,
        CountryID,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerID
            ORDER BY InvoiceDate DESC
        ) AS rn
    FROM fact_sales
    WHERE
        IsRevenue = TRUE
        AND CustomerID != -1
)
UPDATE dim_customer dc
SET RecentCountryID = rc.CountryID
FROM recent_country rc
WHERE
    dc.CustomerID = rc.CustomerID
    AND rc.rn = 1;
''')

No Need Since It Is Defined In dim_product

In [ ]:
con.execute('''ALTER TABLE fact_sales
DROP COLUMN StockCode;''')


Change DataType For CustomerID In fact_sales And dim_customer From Decimal To Integer

In [ ]:
con.execute('''SELECT COUNT(*)  -- To Check for non-integer CustomerIDs
FROM fact_sales
WHERE CustomerID <> CAST(CustomerID AS INT);''').fetch_df()

In [ ]:
con.execute('''ALTER TABLE fact_sales
ALTER COLUMN CustomerID TYPE INT;''')

In [ ]:
con.execute('''ALTER TABLE dim_customer
ALTER COLUMN CustomerID TYPE INT;''')

Save Everything As Database File

In [ ]:
# Save as a database file 
con.execute("CHECKPOINT;") 
con.close()

## Data Quality & Cleaning

### Observations
- Flagged non-revenue transactions with `UnitPrice = 0`
- Identified anonymous customers (missing `CustomerID`)
- Detected bulk transactions indicating wholesale behavior
- Found inconsistent stock codes with multiple descriptions
- Observed customers associated with multiple countries

### Actions Taken
- Excluded non-revenue transactions from revenue calculations
- Removed anonymous customers from customer-level analysis
- Flagged bulk transactions for retail vs wholesale segmentation
- Standardized stock descriptions using the most recent value
- Standardized customer country using the most recent residence
